In [22]:
print("Done")

Done


In [23]:
%pwd

'c:\\Projects\\Medical-Chatbot\\research'

In [24]:
import os
os.chdir("../")

In [25]:
%pwd

'c:\\Projects\\Medical-Chatbot'

In [26]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [27]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [28]:
extracted_data = load_pdf_files("data")

In [29]:
len(extracted_data)

2053

In [30]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [31]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [32]:
minimal_docs

[Document(metadata={'source': 'data\\Healthcare.pdf'}, page_content='Healthcare\nPage 1\nPatient_ID Age Gender\n1 29 Male\n2 76 Female\n3 78 Male\n4 58 Other\n5 55 Female\n6 49 Male\n7 69 Male\n8 25 Other\n9 11 Male\n10 47 Male\n11 78 Other\n12 72 Male\n13 52 Female\n14 64 Female\n15 34 Other\n16 12 Male\n17 88 Female\n18 33 Other\n19 69 Female\n20 34 Other\n21 82 Other\n22 68 Male\n23 40 Male\n24 9 Other\n25 34 Other\n26 40 Female\n27 3 Other\n28 81 Male\n29 66 Male\n30 32 Female\n31 53 Female\n32 52 Other\n33 18 Female\n34 71 Male\n35 62 Male\n36 59 Female\n37 75 Other\n38 75 Female\n39 87 Male\n40 80 Male\n41 34 Female\n42 80 Other\n43 17 Female\n44 57 Other\n45 71 Female\n46 71 Male\n47 34 Other\n48 36 Male\n49 57 Other\n50 5 Female\n51 27 Other\n52 80 Other'),
 Document(metadata={'source': 'data\\Healthcare.pdf'}, page_content='Healthcare\nPage 2\n53 23 Other\n54 32 Female\n55 29 Male\n56 25 Female\n57 66 Female\n58 14 Other\n59 15 Female\n60 69 Other\n61 80 Female\n62 86 Female\n

In [33]:
len(minimal_docs)

2053

In [34]:
def text_splitter(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500, 
        chunk_overlap=20
    )
    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk

In [35]:
text_chunk = text_splitter(minimal_docs)
print(f"Number of chunks: {len(text_chunk)}")

Number of chunks: 10938


In [36]:
text_chunk

[Document(metadata={'source': 'data\\Healthcare.pdf'}, page_content='Healthcare\nPage 1\nPatient_ID Age Gender\n1 29 Male\n2 76 Female\n3 78 Male\n4 58 Other\n5 55 Female\n6 49 Male\n7 69 Male\n8 25 Other\n9 11 Male\n10 47 Male\n11 78 Other\n12 72 Male\n13 52 Female\n14 64 Female\n15 34 Other\n16 12 Male\n17 88 Female\n18 33 Other\n19 69 Female\n20 34 Other\n21 82 Other\n22 68 Male\n23 40 Male\n24 9 Other\n25 34 Other\n26 40 Female\n27 3 Other\n28 81 Male\n29 66 Male\n30 32 Female\n31 53 Female\n32 52 Other\n33 18 Female\n34 71 Male\n35 62 Male\n36 59 Female\n37 75 Other\n38 75 Female\n39 87 Male'),
 Document(metadata={'source': 'data\\Healthcare.pdf'}, page_content='39 87 Male\n40 80 Male\n41 34 Female\n42 80 Other\n43 17 Female\n44 57 Other\n45 71 Female\n46 71 Male\n47 34 Other\n48 36 Male\n49 57 Other\n50 5 Female\n51 27 Other\n52 80 Other'),
 Document(metadata={'source': 'data\\Healthcare.pdf'}, page_content='Healthcare\nPage 2\n53 23 Other\n54 32 Female\n55 29 Male\n56 25 Female\

In [37]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\21mkc\AppData\Local\Temp\ipykernel_21132\2019898946.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [38]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [39]:
vector = embedding.embed_query("Hello WOrld")
vector

[-0.034477248787879944,
 0.031023213639855385,
 0.006734968163073063,
 0.02610895223915577,
 -0.03936203941702843,
 -0.16030244529247284,
 0.06692398339509964,
 -0.006441461853682995,
 -0.0474504716694355,
 0.014758830890059471,
 0.07087528705596924,
 0.05552760139107704,
 0.019193364307284355,
 -0.026251312345266342,
 -0.010109573602676392,
 -0.026940463110804558,
 0.022307418286800385,
 -0.02222663164138794,
 -0.149692565202713,
 -0.017493005841970444,
 0.007676235865801573,
 0.05435224995017052,
 0.003254480427131057,
 0.0317259207367897,
 -0.0846213772892952,
 -0.02940598502755165,
 0.051595576107501984,
 0.048124026507139206,
 -0.0033147847279906273,
 -0.05827918276190758,
 0.04196928068995476,
 0.02221066877245903,
 0.1281888484954834,
 -0.02233896031975746,
 -0.011656244285404682,
 0.06292830407619476,
 -0.03287629783153534,
 -0.09122602641582489,
 -0.03117540292441845,
 0.05269957706332207,
 0.04703482240438461,
 -0.0842030718922615,
 -0.03005615808069706,
 -0.02074484899640083

In [40]:
print("Vector length:", len(vector))

Vector length: 384


In [41]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [42]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [43]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [44]:
pc

In [45]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,
        metric= "cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [46]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    
    index_name=index_name
)

In [ ]:
# Loading Existing index

from langchain_pinecone import PineconeSparseVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

Add more data to the existing Pinecone index

In [48]:
docser = Document(
    page_content="This project is called as a Medbot.",
    metadata={"source":"Youtube"}
)

In [50]:
docsearch.add_documents(documents=[docser])

['3726fd29-97c6-43ee-9004-1e4522fe3bf3']

In [51]:
retriver = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [52]:
retrieved_docs = retriver.invoke("What is Acne?")
retrieved_docs

[Document(id='11797825-1291-4bf5-bf86-140b20ac2bd1', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='45d9b243-1188-4798-92eb-fbaa76286549', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='7c189164-08bc-42a3-9887-897332d761e8', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25')]

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

c:\Users\21mkc\anaconda3\envs\medbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()

True

In [63]:
chatModel = ChatGroq(
    model="llama-3.3-70b-versatile",
    max_retries=5,
    timeout=60,
)

In [ ]:
%pip install pydantic

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys
print(sys.executable)

c:\Users\21mkc\anaconda3\envs\medbot\python.exe


In [64]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# Initialize the model to test connection
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [ ]:
import langchain
print(langchain.__version__)

0.3.25


In [65]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that please "
    "consult a doctor. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [66]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriver, question_answer_chain)

In [57]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [67]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response["answer"])

Acne is a skin disorder in which the sebaceous glands become inflamed. It is a general term used to describe this condition, with Acne vulgaris being a common form that affects the face. Acne is characterized by inflammation of the sebaceous glands, leading to various skin issues.


In [68]:
response = rag_chain.invoke({"input": "I have backpain, what should i do?"})
print(response["answer"])

You should consult a doctor to determine the cause of your back pain, as it can be a symptom of various conditions. A doctor can evaluate your symptoms and provide a proper diagnosis and treatment plan. Please consult a doctor for personalized advice on managing your back pain.


In [69]:
response = rag_chain.invoke({"input": "what is the Treatement on Acne?"})
print(response["answer"])

The treatment for acne depends on its severity and may include topical drugs such as tretinoin, benzoyl peroxide, or salicylic acid. For mild non-inflammatory acne, reducing comedone formation with these topical treatments is effective. Improvement is usually seen in two to four weeks, and topical antibiotics may be added for inflammatory cases.
